# 6 - FINE-TUNE A CROSS-ENCODER RERANKER on this corpus

`cross-encoder/ms-marco-MiniLM-L-6-v2` (22M), trained on hard negatives mined by the
actual retriever. **MiniLM deliberately, not the strongest model:**

| model | zero-shot R@1 (n=100 sample) | pairs/s | full-corpus eval |
|---|---|---|---|
| bge bi-encoder | 40.0% | - | - |
| **MiniLM-L6** | 42.0% (+2.0, ~nothing) | **1,332** | **20 min** |
| bge-reranker-v2-m3 | 47.0% (+7.0) | 76 | 357 min |

- MiniLM gains almost nothing zero-shot, so any large jump here is **unambiguously from
  training**, not from a strong base model.
- It is **18x faster**, so if fine-tuning takes it past v2-m3's 47% that is the production
  answer: better *and* cheaper.
- One epoch is ~5 minutes. v2-m3 would be ~80 minutes an epoch.

**Decision rule, set before the run:** if val R@5 clears ~85% (v2-m3's zero-shot level),
stop and ship MiniLM. If it stalls around 78-80%, fine-tune v2-m3 or distil v2-m3's
rankings into MiniLM.

## Why a split is mandatory from here

This run **selects** - epochs, learning rate, negatives per question, which checkpoint. Do
that on the questions you report on and the number is inflated by an unknown amount, and
unlike the cleaning defects it cannot be repaired afterwards. `step4-rerank/split.json`
holds **14,603 train / 1,626 val**, stratified by question type and **grouped by page** so
no page's chunks appear on both sides.

## The failure mode this notebook guards hardest against

A sampled "negative" that is actually a gold chunk for that question trains the model to
rank the right answer **down**. Nothing would crash; the metrics would just be quietly
worse. §3 removes every gold chunk from the negative pool and asserts the intersection is
empty.

In [ ]:
# §0 SETUP
import os, sys, json, pickle, time, gc, math, random
sys.path.insert(0, os.getcwd())
import numpy as np
import torch
import torch.nn as nn
import nqrun
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_scheduler)

In [ ]:
# §1 CONFIG
BASE     = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
N_NEG    = 7           # negatives sampled per question per epoch -> 8 pairs each
EPOCHS   = 3
LR       = 2e-5
WARMUP   = 0.1         # fraction of total steps
BATCH_Q  = 8           # questions per step -> BATCH_Q * (1 + N_NEG) pairs per forward
MAXLEN   = 320         # query ~15 tok + chunk median 227 (notebook 3) + specials
EVAL_BS  = 128
SEED     = 0
NEG_FROM = 100         # sample negatives from the retriever's top-100 (hard negatives)
KS       = (1, 3, 5, 10, 20, 50, 100)
VAL_SUB  = None        # None = all 1,626 val questions; set an int for a faster loop

# zero-shot references on the SAME val questions are computed in §5 - these are the
# the n=100 sample figures, kept only as a sanity anchor.
REF_SAMPLE = {'bi_r@1': 0.400, 'bi_r@5': 0.750,
          'minilm_r@1': 0.420, 'minilm_r@5': 0.740,
          'v2m3_r@1': 0.470, 'v2m3_r@5': 0.850}
TARGET_R5 = 0.85       # the decision rule

CHUNK, OVERLAP, STRIDE = 200, 50, 150
MAXT = 5
N_PAGES_EXPECTED, N_QS_EXPECTED, N_CHUNKS_EXPECTED = 18997, 16229, 469387
OUT, HF_DIR = nqrun.art('step4-rerank'), 'step2-retrieval'
CKPT = OUT / 'minilm-ft'
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
print(f'{BASE} | {EPOCHS} epochs | {N_NEG} negs/q | lr {LR} | batch {BATCH_Q} q '
      f'= {BATCH_Q * (1 + N_NEG)} pairs')
print(f'root {nqrun.ROOT} | run {nqrun.RUN_ID} | {nqrun.gpu_info()}')

## Definitions

Nothing below executes; running these cells only defines names.

In [ ]:
# DEFINITIONS
#
# The loss is a listwise cross-entropy over one positive and N_NEG negatives:
# the model must score the positive above its OWN negatives, which is what a
# reranker does at inference. A plain per-pair binary loss optimises calibration
# instead of ordering, and ordering is the job.
#
# evaluate() marks a question UNREACHABLE when its gold chunk is not in the pool
# at all, and never credits it — an earlier two-level cascade reported 98.1%
# against a 96.1% ceiling before
# that was fixed.
def chunks_for_span(p, lo, hi):
    """Global chunk ids on page p that wholly contain the word span [lo, hi)."""
    j0 = max(0, -(-(hi - CHUNK) // STRIDE))
    j1 = min(lo // STRIDE, n_per_page[p] - 1)
    return [int(base[p] + j) for j in range(j0, j1 + 1)]

def score_pairs(qs, cs, train=True):
    """Cross-encoder logit for each (query, chunk) pair."""
    enc = tok(qs, cs, padding=True, truncation=True, max_length=MAXLEN,
              return_tensors='pt').to(dev)
    with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=(dev == 'cuda')):
        out = model(**enc).logits.squeeze(-1)
    return out

def batch_loss(qis, rng):
    """(B, 1+N_NEG) scores; the positive is always column 0, so the label is always 0."""
    qs, cs = [], []
    for i in qis:
        pos = POS[i][rng.integers(len(POS[i]))]
        neg = rng.choice(NEGPOOL[i], size=N_NEG, replace=False)
        qs += [qtext[i]] * (1 + N_NEG)
        cs += [texts[pos]] + [texts[int(c)] for c in neg]
    s = score_pairs(qs, cs).view(len(qis), 1 + N_NEG)
    return nn.functional.cross_entropy(s.float(),
                                       torch.zeros(len(qis), dtype=torch.long, device=dev))

@torch.no_grad()
def evaluate(idx):
    """Rerank each question's top-100 and return (ranks, recall@k, reachable)."""
    model.eval()
    UNREACHABLE = 10 ** 9
    ranks = np.full(len(idx), UNREACHABLE, dtype='int64')
    for n, i in enumerate(idx):
        pool = top100[i]
        gp = [j for j, c in enumerate(pool.tolist()) if c in gold_sets[i]]
        if not gp:
            continue
        sc = []
        for a in range(0, len(pool), EVAL_BS):
            blk = pool[a:a + EVAL_BS]
            sc.append(score_pairs([qtext[i]] * len(blk),
                                   [texts[int(c)] for c in blk]).float().cpu().numpy())
        order = np.argsort(-np.concatenate(sc))
        ranks[n] = min(int(np.where(order == j)[0][0]) for j in gp)
    model.train()
    reach = float((ranks < UNREACHABLE).mean())
    return ranks, {k: float((ranks < k).mean()) for k in KS}, reach

## Orchestration

In [ ]:
# §2 DATA + GUARDS
IDX = json.load(open(nqrun.hf_pull(f'{HF_DIR}/chunk_index.json')))
H, owner = IDX['doc_hashes'], np.array(IDX['owner'], dtype='int32')
assert IDX['chunk'] == CHUNK and IDX['overlap'] == OVERLAP, 'chunking differs from notebook 3'
assert len(H) == N_PAGES_EXPECTED and len(owner) == N_CHUNKS_EXPECTED
n_per_page = np.bincount(owner, minlength=len(H)).astype('int64')
base = np.concatenate([[0], np.cumsum(n_per_page)[:-1]]).astype('int64')
top100 = np.load(nqrun.hf_pull(f'{HF_DIR}/top100_chunks.npy'))
assert top100.shape == (N_QS_EXPECTED, 100)

SP = json.load(open(nqrun.hf_pull('step4-rerank/split.json')))
train_idx = np.array(SP['train_idx']); val_idx = np.array(SP['val_idx'])
assert SP['n_questions'] == N_QS_EXPECTED, 'split was built on a different corpus'
assert not (set(train_idx.tolist()) & set(val_idx.tolist())), 'split overlaps'
qtype = np.array(SP['question_type'])

C = pickle.load(open(nqrun.hf_pull('step1-clean/clean_corpus.pkl'), 'rb'))
docs, questions = C['documents'], C['questions']
assert len(docs) == N_PAGES_EXPECTED and len(questions) == N_QS_EXPECTED, 'WRONG ARTIFACT'
assert list(docs) == H, 'page order differs from the chunk matrix'
row = {h: i for i, h in enumerate(H)}

texts = []
for h in H:
    w, title = docs[h]['words'], docs[h]['title']
    for a in range(0, max(len(w), 1), STRIDE):
        part = w[a:a + CHUNK]
        if not part:
            break
        texts.append(f"{title}. {' '.join(part)}")
assert len(texts) == N_CHUNKS_EXPECTED, f'rebuilt {len(texts)}'

gold_sets = [{c for _, lo, hi in q['gold']
              for c in chunks_for_span(row[q['doc_hash']], lo, hi)} for q in questions]
qtext = [q['question'] for q in questions]

bad = 0
for i in np.random.default_rng(0).choice(len(questions), 300, replace=False):
    q = questions[i]
    for k, (_, lo, hi) in enumerate(q['gold']):
        for c in chunks_for_span(row[q['doc_hash']], lo, hi):
            if q['answers_rendered'][k] not in texts[c] or owner[c] != row[q['doc_hash']]:
                bad += 1
print(f'alignment check (300 sampled): {bad} mismatches  <- must be 0')
assert bad == 0, 'rebuilt chunk text does not align with the saved embedding order'
print(f'train {len(train_idx):,} | val {len(val_idx):,} | {len(texts):,} chunks | guards OK')
del C, docs; gc.collect()

In [ ]:
# §3 TRAINING EXAMPLES. One positive + N_NEG hard negatives per question, negatives drawn
# from the retriever's top-100 with EVERY gold chunk removed first.
#
# This is the guard that matters. A negative that is secretly a gold chunk teaches the
# model to rank the correct passage DOWN - no crash, no warning, just quietly worse
# metrics. Note gold_sets can hold several chunks per question (overlapping windows, and
# up to 8 for multi-answer list questions), so excluding one is not enough.
POS, NEGPOOL, skipped = {}, {}, []
for i in train_idx.tolist():
    pool = top100[i].tolist()
    g = gold_sets[i]
    pos = [c for c in pool[:NEG_FROM] if c in g]
    neg = [c for c in pool[:NEG_FROM] if c not in g]
    if not pos or len(neg) < N_NEG:
        skipped.append(i)                 # gold not retrieved, or too few negatives
        continue
    POS[i] = pos
    NEGPOOL[i] = neg

usable = np.array(sorted(POS))
print(f'usable train questions: {len(usable):,} of {len(train_idx):,}'
      f'  (skipped {len(skipped):,}: gold outside the top-{NEG_FROM})')
print(f'negatives available per question: min {min(len(v) for v in NEGPOOL.values())} | '
      f'median {int(np.median([len(v) for v in NEGPOOL.values()]))}')

leak = sum(1 for i in usable.tolist() if set(NEGPOOL[i]) & gold_sets[i])
print(f'negatives that are secretly gold: {leak}        <- must be 0')
assert leak == 0, 'a gold chunk is in the negative pool - training would invert the signal'

print('\nusable train questions by type (training needs enough of each):')
for t in sorted(set(qtype[usable].tolist())):
    print(f'  {t:<8} {int((qtype[usable] == t).sum()):>6,}')

steps_per_epoch = math.ceil(len(usable) / BATCH_Q)
total_steps = steps_per_epoch * EPOCHS
print(f'\n{steps_per_epoch:,} steps/epoch x {EPOCHS} = {total_steps:,} steps | '
      f'{len(usable) * (1 + N_NEG):,} pairs/epoch')

In [ ]:
# §4 MODEL + LOSS.
# Listwise cross-entropy over one positive and N_NEG negatives: the model must score the
# positive above its OWN negatives, which is what a reranker does at inference. A plain
# per-pair binary loss optimises calibration instead of ordering, and ordering is the job.

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=1).to(dev)
print(f'{BASE} | {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params | {dev}')

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = get_scheduler('linear', opt, num_warmup_steps=int(WARMUP * total_steps),
                      num_training_steps=total_steps)

val_eval = val_idx if VAL_SUB is None else np.random.default_rng(0).choice(
    val_idx, VAL_SUB, replace=False)
print(f'evaluating on {len(val_eval):,} val questions x 100 chunks = '
      f'{len(val_eval) * 100 / 1000:.0f}k pairs per eval')

In [ ]:
# §5 BASELINE then TRAIN. The zero-shot row is measured on THESE val questions, so the
# comparison is like-for-like rather than against the earlier 100-question sample.
t0 = time.time()
_, base_rec, reach = evaluate(val_eval)
print(f'ceiling (gold in the top-100 pool): {reach:.1%}   <- no row can exceed this')
print(f"epoch 0 (zero-shot)  " + '  '.join(f'R@{k} {base_rec[k]:.1%}' for k in KS)
      + f'   [{time.time() - t0:.0f}s]')

history = [{'epoch': 0, 'recall': base_rec, 'loss': None}]
best = {'epoch': 0, 'r5': base_rec[5], 'state': None}
rng = np.random.default_rng(SEED)
model.train()

for ep in range(1, EPOCHS + 1):
    perm = rng.permutation(usable)
    run, t0 = 0.0, time.time()
    for st in range(steps_per_epoch):
        qis = perm[st * BATCH_Q:(st + 1) * BATCH_Q].tolist()
        if not qis:
            break
        loss = batch_loss(qis, rng)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        run += loss.item()
        if st and st % 300 == 0:
            el = time.time() - t0
            print(f'  ep{ep} step {st:,}/{steps_per_epoch:,} loss {run / (st + 1):.4f} '
                  f'({el / (st + 1) * steps_per_epoch / 60:.1f} min/epoch)', flush=True)
    tr_loss = run / max(steps_per_epoch, 1)

    _, rec, _ = evaluate(val_eval)
    history.append({'epoch': ep, 'recall': rec, 'loss': tr_loss})
    print(f'epoch {ep} loss {tr_loss:.4f}  ' + '  '.join(f'R@{k} {rec[k]:.1%}' for k in KS)
          + f'   [{time.time() - t0:.0f}s]', flush=True)
    if rec[5] > best['r5']:                      # checkpoint selection on val R@5
        best = {'epoch': ep, 'r5': rec[5],
                'state': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}
        print(f'    new best val R@5 {rec[5]:.1%} (epoch {ep})')

print(f"\nbest epoch {best['epoch']} at val R@5 {best['r5']:.1%}")
if best['state'] is not None:
    model.load_state_dict(best['state'])

In [ ]:
# §6 REPORT + the decision rule, stated before the run.
print(f"{'epoch':>6}{'loss':>9}" + ''.join(f'{("R@" + str(k)):>8}' for k in KS))
for h in history:
    ls = f"{h['loss']:.4f}" if h['loss'] is not None else 'zero-shot'
    print(f"{h['epoch']:>6}{ls:>9}" + ''.join(f"{h['recall'][k]:>8.1%}" for k in KS))

z, f = history[0]['recall'], history[best['epoch']]['recall']
print('\nGAIN FROM FINE-TUNING (same val questions, zero-shot -> best epoch)')
for k in (1, 3, 5, 10):
    print(f'  R@{k:<4} {z[k]:>6.1%} -> {f[k]:>6.1%}   {(f[k] - z[k]) * 100:>+5.1f} pts')

print('\nCEILING CHECK')
ok = f[max(KS)] <= reach + 1e-9
print(f'  R@{max(KS)} {f[max(KS)]:.1%} vs ceiling {reach:.1%}  '
      f"{'OK' if ok else 'VIOLATED - BUG'}")

print('\nDECISION RULE (set before the run)')
print(f'  val R@5 = {f[5]:.1%} | target {TARGET_R5:.0%} '
      f"(bge-reranker-v2-m3 zero-shot, n=100)")
if f[5] >= TARGET_R5:
    print('  -> SHIP MiniLM: it matches or beats v2-m3 at 18x the speed '
          '(38 ms vs 660 ms at k=50).')
else:
    print(f'  -> {TARGET_R5 - f[5]:.1%} short. Next: fine-tune v2-m3 (~5-6 h, ~$3.50) or '
          'distil v2-m3 rankings into MiniLM.')
print('  CAVEAT: the 85% target came from n=100 (SE ~5 pts); this val set is '
      f'{len(val_eval):,} questions, so these numbers are the more reliable ones.')

print('\nPER-TYPE val R@5 (stratification exists so this is readable)')
ranks_f, _, _ = evaluate(val_eval)
vt = qtype[val_eval]
for t in sorted(set(vt.tolist())):
    m = vt == t
    if m.sum() >= 20:
        print(f'  {t:<8} n={int(m.sum()):>5,}  R@5 {float((ranks_f[m] < 5).mean()):>6.1%}')
    else:
        print(f'  {t:<8} n={int(m.sum()):>5,}  (too few to report)')

In [ ]:
# §7 SAVE the best checkpoint plus the training record.
CKPT.mkdir(parents=True, exist_ok=True)
model.save_pretrained(CKPT); tok.save_pretrained(CKPT)
size = sum(f.stat().st_size for f in CKPT.rglob('*') if f.is_file()) / 1e6
print(f'-> {CKPT}  ({size:.0f} MB)')

json.dump({'base': BASE, 'config': {'n_neg': N_NEG, 'epochs': EPOCHS, 'lr': LR,
                                    'batch_q': BATCH_Q, 'maxlen': MAXLEN, 'seed': SEED},
           'n_train_usable': int(len(usable)), 'n_train_skipped': len(skipped),
           'n_val': int(len(val_eval)), 'ceiling': reach,
           'best_epoch': best['epoch'],
           'history': [{'epoch': h['epoch'], 'loss': h['loss'],
                        'recall': {str(k): v for k, v in h['recall'].items()}}
                       for h in history]},
          open(OUT / 'finetune_history.json', 'w'), indent=1)
nqrun.hf_push(str(OUT / 'finetune_history.json'), 'step4-rerank/finetune_history.json',
              f'{BASE} fine-tune history, best epoch {best["epoch"]}')
nqrun.hf_push(str(CKPT), 'step4-rerank/minilm-ft', f'{BASE} fine-tuned on this corpus')

nqrun.log_metrics('6_finetune_rerank', base=BASE, n_neg=N_NEG, epochs=EPOCHS, lr=LR,
                  n_train_usable=int(len(usable)), n_val=int(len(val_eval)),
                  ceiling=reach, best_epoch=best['epoch'],
                  **{f'zeroshot_r@{k}': history[0]['recall'][k] for k in KS},
                  **{f'finetuned_r@{k}': history[best['epoch']]['recall'][k] for k in KS})